# Imports and data loading #

In [13]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import time

%reload_ext autoreload
%autoreload 2

In [16]:
from src.helpers import *
from src.preprocessing import *
from src.model import *

In [7]:
DATA_PATH = "./src/dataset/"
SUB_SAMPLE = False

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH, SUB_SAMPLE)
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)

(328135, 321)
(328135,)
(109379, 321)


In [9]:
with open(os.path.join(DATA_PATH, "x_train.csv")) as f:
    header = f.readline().strip().split(",")
header = np.array(header[1:]) # Remove ID
print("Headers:", len(header))
#print(header)

Headers: 321


# Preprocessing #

In [10]:
x_tr, header_tr, fitted_log_names, onehot_cols = prepare(x_train, header)
x_te, _, _, _ = prepare(x_test,  header, fitted_log_names, onehot_cols)
y = (y_train == 1).astype(int) # Change label -1 -> 0, +1 -> +1

# Training #

## I. Logistic regression

Logistic regression alone with mean pooling achieved a F1-score of 0.452 on the test set.

In [14]:
def cv_logreg_test(X, y, X_test, header, lam=0.002, k=5, seed=0):
    folds = stratified_kfold_indices(y, k, seed)
    oof = np.zeros(len(y))
    pte = np.zeros(len(X_test))
    for i in range(k):
        va = folds[i]; tr = np.concatenate([folds[j] for j in range(k) if j != i])
        pp = fit_preprocess(X[tr])
        Xa, na = apply_fitted_preprocess(X[tr], pp, header)
        Xb, _ = apply_fitted_preprocess(X[va], pp, header)
        Xt, _ = apply_fitted_preprocess(X_test, pp, header)
        w = newton_fit(Xa, y[tr], lam, n_iter=10, tol=1e-6)
        oof[va] = sigmoid(Xb @ w)
        pte += sigmoid(Xt @ w) / k
    return oof, pte

oof_l, te_l = cv_logreg_test(x_tr, y, x_te, header_tr)

## II. MLP

In [ ]:
def cv_mlp(X, y, X_test, header, k=5, seed=0, load_P=False, **kw):
    folds = stratified_kfold_indices(y, k, seed)
    oof = np.zeros(len(y))
    pte = np.zeros(len(X_test))

    for i in range(k):
        t0 = time.time()
        va = folds[i]
        tr = np.concatenate([folds[j] for j in range(k) if j != i])
        pp = fit_preprocess(X[tr])
        Xa, na = apply_fitted_preprocess(X[tr], pp, header)
        Xb, _ = apply_fitted_preprocess(X[va], pp, header)
        Xt, _ = apply_fitted_preprocess(X_test, pp, header)

        if(load_P):
            P = np.load(f"./src/load/MLP_P_{i}.npy", allow_pickle=True)
        else:
            P, _ = train_mlp(Xa, y[tr], seed=seed * 100 + i, **kw)
            np.save(f"./src/load/MLP_P_{i}", np.array(P, dtype=object))

        oof[va] = predict(P, Xb)
        pte += predict(P, Xt) / k
        print(f"pli {i + 1}/{k} | {time.time() - t0:.0f} s", flush=True)
    return oof, pte

oof_m, te_m = cv_mlp(x_tr, y, x_te, header_tr, epochs=15)

pli 1/5 | 105 s
pli 2/5 | 107 s
pli 3/5 | 104 s
pli 4/5 | 100 s
pli 5/5 | 101 s


## III. Merged model
Here we combine the MLP with the logistic regression to try to get the best of both worlds.

In [19]:
for a in (0.0, 0.25, 0.5, 0.75, 1.0):  # a = weight of MLP vs log reg
    report(y, a * oof_m + (1 - a) * oof_l, f"a = {a:.2f}")

a = 0.00     F1 max lisse = 0.4399 au seuil 0.205 (theorie F*/2 = 0.220) | AP = 0.4281
a = 0.25     F1 max lisse = 0.4430 au seuil 0.215 (theorie F*/2 = 0.222) | AP = 0.4308
a = 0.50     F1 max lisse = 0.4442 au seuil 0.215 (theorie F*/2 = 0.222) | AP = 0.4323
a = 0.75     F1 max lisse = 0.4448 au seuil 0.225 (theorie F*/2 = 0.222) | AP = 0.4325
a = 1.00     F1 max lisse = 0.4443 au seuil 0.225 (theorie F*/2 = 0.222) | AP = 0.4310


In [24]:
a=0.7
t_star = 0.225
pred = np.where(a * te_m + (1 - a) * te_l >= t_star, 1, -1)
create_csv_submission(test_ids, pred, "submission_demo.csv")